# ARES: Fast Evaluation for Remaining Benchmark Domains (Qwen2.5-7B-Instruct 4-bit)

This dedicated notebook loads the pre-trained checkpoints (GRM, LRM, Router, 5 LoRA Experts) and evaluates the remaining benchmark domains (**GSM8K Math, MBPP Code, AI2-ARC Science, CommonsenseQA Reasoning**).

- Automatically resumes from `outputs/benchmark_checkpoint_7b_test_500.json` (skips the already completed 450 WikiText General samples).
- Uses unbuffered stdout (`!python -u`) for live real-time log streaming.
- Completes in ~1 hour on dual T4 GPU without any timeout risk.

In [ ]:
# === [1/3] Environment Setup & Sync Repository ===
!if [ -d "/kaggle/working/ARES-research" ]; then cd /kaggle/working/ARES-research && git pull origin main; else cd /kaggle/working && git clone https://github.com/sharksurfauto-byte/ARES-research.git; fi

%cd /kaggle/working/ARES-research
!pip install -q --upgrade pip
!pip install -q -e .

import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("=== Environment Ready ===")

In [ ]:
# === [2/3] Checkpoint Linking & Progress Resumption Setup ===
import os, shutil
from pathlib import Path

print("Searching for pre-trained checkpoints across /kaggle/input/ ...")

# 1. Search recursively for grm.pt anywhere under /kaggle/input/
grm_candidates = list(Path("/kaggle/input").rglob("grm.pt"))
print(f"Found GRM candidate files: {grm_candidates}")

if grm_candidates:
    ckpt_source = grm_candidates[0].parent.parent
    print(f"[Found Checkpoints] Copying entire directory from {ckpt_source} to ./checkpoints/ ...")
    shutil.copytree(ckpt_source, "checkpoints", dirs_exist_ok=True)
else:
    for fb in [
        Path("/kaggle/input/datasets/aliasgharjjawadwala/ares-eval-input/checkpoints"),
        Path("/kaggle/input/ares-eval-input/checkpoints"),
    ]:
        if fb.exists():
            shutil.copytree(fb, "checkpoints", dirs_exist_ok=True)
            break

# 2. Search recursively for benchmark_checkpoint_7b_test_500.json
ckpt_json_candidates = list(Path("/kaggle/input").rglob("benchmark_checkpoint_7b_test_500.json"))
print(f"Found checkpoint JSON candidates: {ckpt_json_candidates}")
if ckpt_json_candidates:
    os.makedirs("outputs", exist_ok=True)
    target_json = Path("outputs/benchmark_checkpoint_7b_test_500.json")
    shutil.copyfile(ckpt_json_candidates[0], target_json)
    print(f"[Found Progress Checkpoint] Copied progress file ({target_json.stat().st_size / (1024*1024):.2f} MB) to ./outputs/")

# 3. Verify critical checkpoints exist
assert Path("checkpoints/reliability/grm.pt").exists(), f"ERROR: grm.pt not found! Checked: {list(Path('checkpoints').rglob('*'))}"
assert Path("checkpoints/reliability/lrm.pt").exists(), "ERROR: lrm.pt not found!"
assert Path("checkpoints/router/router_best.pt").exists() or Path("checkpoints/router/router.pt").exists(), "ERROR: router checkpoint not found!"
print("\nSUCCESS: All pre-trained checkpoints (GRM, LRM, Router, 5 Experts) verified and ready!")


In [ ]:
# === [3/3] Fast Benchmark Evaluation (GSM8K, MBPP, AI2-ARC, CommonsenseQA) ===
# Note: !python -u streams live stdout directly to the notebook log with zero buffering delay

!python -u scripts/run_ares_pipeline.py \
    --model_name "Qwen/Qwen2.5-7B-Instruct" \
    --grm_checkpoint "checkpoints/reliability/grm.pt" \
    --lrm_checkpoint "checkpoints/reliability/lrm.pt" \
    --router_checkpoint "checkpoints/router/router_best.pt" \
    --expert_dir "checkpoints/experts" \
    --benchmark all \
    --split test \
    --samples_per_domain 100 \
    --max_new_tokens 64 \
    --checkpoint_every 20 \
    --checkpoint_file "outputs/benchmark_checkpoint_7b_test_500.json" \
    --run_baselines \
    --output_report "benchmarks_ares_report.md" \
    --output_json "benchmarks_ares_results.json" \
    --device cuda

print("\n" + "=" * 70)
print("=== ARES BENCHMARK EVALUATION SUMMARY REPORT ===")
print("=" * 70)
!cat benchmarks_ares_report.md